# 撤稿论文引用分析 — 模块(一): 找到 Focal Paper (i)

**日期:** 2026-05-12
**目标:** 从全量引用关系中筛出引用了撤稿文献的论文
**策略:** DuckDB 直接 join 52G CSV，不加载进 pandas
**预计耗时:** 5-10 分钟

In [1]:
import duckdb
import pandas as pd
import time
import os

con = duckdb.connect()
con.execute("SET memory_limit = '100GB'")
print("DuckDB 已连接, 内存上限 100G")

DuckDB 已连接, 内存上限 100G


## Step 1/4: 加载撤稿论文名单

matched_output2.csv (50MB, 约4万篇撤稿论文)

In [2]:
retra_path = '/Data4/yutao_wen/matched_output2.csv'

con.execute(f"CREATE TEMP TABLE retra_ids AS SELECT DISTINCT CAST(id AS VARCHAR) AS id FROM read_csv_auto('{retra_path}')")
n_retra = con.execute("SELECT COUNT(*) FROM retra_ids").fetchone()[0]
print(f"撤稿论文总数: {n_retra:,}")

撤稿论文总数: 42,188


## Step 2/4: 匹配

扫描 52G PaperReferences.csv, 只保留 reference_ids 在撤稿名单中的行
PaperReferences.csv 两列: id (论文) 引用 reference_ids (被引文献)

In [3]:
refs_path = '/data6/Data1/DATA/Dimensions2024/20240101/PaperReferences.csv'

print("正在扫描 PaperReferences.csv (52G)... 预计 5-10 分钟")
t0 = time.time()

con.execute(f"CREATE TABLE df_containretra AS SELECT pr.id, pr.reference_ids FROM read_csv_auto('{refs_path}') AS pr WHERE pr.reference_ids IN (SELECT id FROM retra_ids)")

elapsed = time.time() - t0
print(f"完成! 耗时: {elapsed/60:.1f} 分钟")

count = con.execute("SELECT COUNT(*) FROM df_containretra").fetchone()[0]
unique_i = con.execute("SELECT COUNT(DISTINCT id) FROM df_containretra").fetchone()[0]
print(f"匹配行数: {count:,}")
print(f"唯一 focal paper (i): {unique_i:,}")
print(f"平均每个 i 引用撤稿文献数: {count/unique_i:.2f}")

正在扫描 PaperReferences.csv (52G)... 预计 5-10 分钟


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

完成! 耗时: 7.3 分钟
匹配行数: 765,839
唯一 focal paper (i): 647,439
平均每个 i 引用撤稿文献数: 1.18


## Step 3/4: 保存

In [4]:
os.makedirs('/Data4/yutao_wen/processed', exist_ok=True)
output_path = '/Data4/yutao_wen/processed/df_containretra.csv'
con.execute(f"COPY df_containretra TO '{output_path}' (HEADER, DELIMITER ',')")
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"已保存: {output_path} ({size_mb:.1f} MB)")

已保存: /Data4/yutao_wen/processed/df_containretra.csv (21.9 MB)


## Step 4/4: 删除汇报

PaperReferences 全量约 20 亿行。保留条件: reference_ids 属于撤稿名单。
删除: reference_ids 不在撤稿名单的普通引用关系。

## 验证: 预览前 5 行

In [5]:
sample = con.execute("SELECT * FROM df_containretra LIMIT 5").df()
print(sample)
con.close()
print("Step 1 完成!")

               id   reference_ids
0  pub.1033312508  pub.1004699774
1  pub.1128828668  pub.1049161136
2  pub.1047989481  pub.1047248881
3  pub.1135272592  pub.1053060092
4  pub.1022173052  pub.1047817620
Step 1 完成!
